# RAG per l'analisi temporale dei bilanci

Questa versione segue la separazione del backend e mantiene una RAG semplice e tracciabile:

- lettura dei PDF dalla cartella locale `data/`;
- estrazione del testo pagina per pagina e metadati azienda/anno fiscale;
- chunking con collegamento ai chunk precedente e successivo;
- interpretazione della query per espandere i termini finanziari e applicare i filtri;
- retrieval semantico Chroma con filtro metadata;
- una sola generazione vincolata al contesto recuperato.

Sono stati rimossi BM25, RRF/reranking e self-correction: non fanno parte del flusso RAG usato qui.

In [ ]:
%pip install -q -U langchain langchain-text-splitters langchain-chroma chromadb sentence-transformers langchain-huggingface pypdf
print('Librerie installate correttamente!')

## 1. Configurazione, moduli e cartella dati

La cartella `app/`, la cartella `data/` e il notebook devono appartenere allo stesso progetto. Il database Chroma viene creato nella cartella del progetto; i PDF vengono letti esclusivamente da `data/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path('/content')]
PROJECT_ROOT = next(
    (path for path in PROJECT_ROOT_CANDIDATES if (path / 'app').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Cartella app non trovata. Posiziona app/, data/ e il notebook '
        'nella stessa cartella del progetto.'
    )

MODULE_ROOT = PROJECT_ROOT
DATA_FOLDER_CANDIDATES = [PROJECT_ROOT / 'data', PROJECT_ROOT / 'app' / 'data']
DATA_FOLDER = next((path for path in DATA_FOLDER_CANDIDATES if path.is_dir()), None)
if DATA_FOLDER is None:
    raise FileNotFoundError(
        f'Cartella dati non trovata. Cercate: {DATA_FOLDER_CANDIDATES}'
    )

pdf_files = sorted(DATA_FOLDER.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'Nessun PDF trovato in: {DATA_FOLDER}')

if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

DB_PATH = PROJECT_ROOT / 'chroma_db_bilanci'

print(f'Progetto caricato da: {PROJECT_ROOT}')
print(f'PDF trovati in data/: {len(pdf_files)}')

## 2. Ingestion dei PDF

L'estrazione e la normalizzazione sono incapsulate in `app/utils/file_utils.py`, come nel backend. L'anno viene letto prima dal nome del file e, se assente, dal testo.

In [ ]:
from app.utils.file_utils import load_pdf_records

pdf_records = load_pdf_records(DATA_FOLDER)

## 3. Chunking e metadati

Ogni chunk conserva azienda, anno fiscale, sorgente e gli identificativi dei chunk adiacenti. Questo permette al retriever di recuperare il contesto vicino senza introdurre un secondo ranking.

In [ ]:
from app.utils.chunk_utils import chunk_documents

tutti_i_chunk = chunk_documents(pdf_records)

## 4. Embedding e indice Chroma

Il modulo `app/init/init_vector_db.py` gestisce embedding e inserimento dei chunk in batch dentro un unico indice vettoriale locale. I filtri per azienda e anno vengono applicati nella fase di retrieval, prima della generazione.

In [ ]:
from app.config.constants import COLLECTION_NAME
from app.init.init_vector_db import init_vector_db

vector_db_result = init_vector_db(
    documents=tutti_i_chunk,
    db_path=DB_PATH,
    collection_name=COLLECTION_NAME,
    rebuild=False,
)
vector_db = vector_db_result.vector_db
all_documents = vector_db_result.all_documents
company_catalog = sorted({document.metadata['company'] for document in all_documents})
year_catalog = sorted({document.metadata['fiscal_year'] for document in all_documents if document.metadata['fiscal_year']})
print(f'Vector DB: {len(all_documents)} chunk')
print(f'Chunk inseriti: {vector_db_result.inserted_chunks}')
print(f'Aziende: {company_catalog}')
print(f'Anni fiscali disponibili: {year_catalog}')

## 5. Modello Groq

Il modello remoto Groq viene usato per interpretare la query e produrre la risposta, senza scaricare Qwen in locale. L'interpretazione deterministica di azienda e anno resta sempre attiva per proteggere i filtri. Prima di eseguire la cella, configura `GROQ_API_KEY` nell'ambiente del kernel.

In [ ]:
import os

from app.config.constants import GROQ_MODEL_ID
from app.utils.rag.model_selector import load_generation_model

if not os.getenv('GROQ_API_KEY'):
    raise RuntimeError('Imposta GROQ_API_KEY nell ambiente del kernel prima di continuare.')

pipe = load_generation_model(provider='groq', model_id=GROQ_MODEL_ID)
print(f'Modello Groq {GROQ_MODEL_ID} pronto.')

## 6. Query interpretation

La query interpretation conserva il meccanismo richiesto: riconosce aziende e anni, risolve alias/ticker dal catalogo, espande i termini finanziari italiani in inglese e usa il modello solo come supporto per completare l'interpretazione.

In [ ]:
import json

from app.utils.rag.generator import generate_text
from app.utils.rag.query_interpreter import QueryInterpreter

def generation_for_interpreter(messages, max_new_tokens=100):
    return generate_text(pipe, messages, max_new_tokens=max_new_tokens)

query_interpreter = QueryInterpreter(
    company_catalog=company_catalog,
    generation_fn=generation_for_interpreter,
)

interpretation = query_interpreter.interpret(
    'Mi dici i ricavi totali del 2022 fiscale di Intel?'
)
print(json.dumps(interpretation.to_dict(), indent=2, ensure_ascii=False))

## 7. Retrieval e risposta

Il flusso è: interpretazione → filtro metadata → ricerca semantica → chunk precedente/successivo → generazione. Non sono presenti BM25, RRF, reranking della risposta o self-correction.

In [ ]:
import json

from app.rag_pipeline import RAGPipeline

# Entry point riutilizzabile sui componenti già costruiti nelle celle precedenti.
rag = RAGPipeline(
    vector_db=vector_db,
    all_documents=all_documents,
    text_generator=pipe,
    model_id=GENERATION_MODEL_ID,
)

# Inserisci qui la query.
query = 'Who spent more on Research and Development (R&D) in 2022, Tesla or PayPal?'
interpretation, retrieval_result, answer = rag.ask(query, top_k=32)

print('INTERPRETAZIONE')
print(json.dumps(interpretation.to_dict(), indent=2, ensure_ascii=False))
print()
print('DIAGNOSTICA RETRIEVAL')
print(json.dumps(retrieval_result.diagnostics, indent=2, ensure_ascii=False))
print()
print('RISPOSTA FINALE')
print(answer)

## 8. Entry point unico

La query da modificare nel normale flusso del notebook è nella cella di retrieval precedente. Se invece vuoi eseguire tutto con un unico comando, modifica `DEFAULT_QUERY` in `main.py` e avvia `main.py`. L'indice già presente viene riutilizzato e il modello generativo viene chiamato tramite Groq.

In [ ]:
# Modalità alternativa: esecuzione completa tramite main.py.
# Richiede GROQ_API_KEY configurata nell'ambiente del kernel.
# from main import run
# query = 'Quali erano i ricavi totali di Intel nel 2022?'
# interpretation, retrieval_result, answer = run(
#     query=query, db_path=str(DB_PATH), initialize=False,
# )
# print(answer)

## Note operative

- Se `candidate_chunks` è 0, controlla il nome dell'azienda e l'anno presenti nei filename dei PDF.
- Per un confronto tra aziende o anni, la query deve citarli esplicitamente.
- Se un PDF non ha testo estraibile, viene segnalato e non entra nell'indice.
- Per eseguire il notebook, mantieni `app/`, `data/`, `main.py` e il notebook nella stessa cartella del progetto.
- In alternativa puoi modificare `DEFAULT_QUERY` in `main.py` ed eseguire `python main.py`; la cartella dati viene cercata in `data/` e poi in `app/data/`.